# Notebook 02 — Read Quality Control: FastQC Metrics

**Module:** 09 — Genomics and NGS  
**Notebook:** 02 of 16  
**Time estimate:** 75 minutes

> The first step of every NGS pipeline is QC. Know what each FastQC metric
> means, what a failing metric implies, and what to do about it.

---
## Step 1 — Motivation

Garbage in, garbage out. A variant calling run on unchecked reads will call adapter
sequences as variants, confuse low-quality 3' bases for SNPs, and fail to align
reads with contamination. QC is not optional — it's the difference between a result
and a false result.

---
## Step 2 — Intuition

FastQC runs a battery of checks on a FASTQ file and produces a report. Key metrics:

1. **Per-base sequence quality:** Q score at each read position. Should be Q30+ in
   the first half of reads; often drops at the 3' end.
2. **Per-sequence quality distribution:** Most reads should have mean Q > 30.
3. **GC content distribution:** Should match the expected GC of the organism.
   Deviations suggest contamination or library bias.
4. **Sequence duplication level:** High duplication = low library complexity or
   optical duplicates. Worrying if >50% of reads are duplicates.
5. **Adapter content:** Illumina TruSeq, Nextera, etc. Should be absent after
   trimming.
6. **Overrepresented sequences:** Adapter, rRNA, or contamination if they appear
   frequently.

---
## Step 3 — Biological Background

**Why does quality drop at 3' ends?**  
In Illumina sequencing-by-synthesis, the base-calling fluorescence signal degrades
as more cycles accumulate: phasing (some clusters get ahead or behind), signal loss,
and increased cross-talk between channels. This is a sequencing chemistry effect, not
a library quality issue.

**What is adapter contamination?**  
If the DNA insert is shorter than the read length, the sequencer reads into the
adapter ligated to the other end of the fragment. These adapter sequences align
nowhere in the genome and cause alignment failure.

**Trimmomatic parameters:**
- `LEADING:3` — remove leading bases with Q < 3
- `TRAILING:3` — remove trailing bases with Q < 3
- `SLIDINGWINDOW:4:15` — cut when 4-base window drops below Q15
- `MINLEN:36` — discard reads shorter than 36 bp after trimming
- `ILLUMINACLIP` — adapter sequence removal

In [ ]:
# Step 6 — Python: Implement a FastQC-like quality report
import numpy as np
import math
from collections import Counter
from dataclasses import dataclass
from typing import Iterator


@dataclass
class FastqRecord:
    header: str
    sequence: str
    quality: str

    @property
    def phred_scores(self) -> list[int]:
        return [ord(c) - 33 for c in self.quality]

    @property
    def mean_quality(self) -> float:
        scores = self.phred_scores
        return sum(scores) / len(scores)

    @property
    def gc_content(self) -> float:
        gc = self.sequence.upper().count('G') + self.sequence.upper().count('C')
        return gc / len(self.sequence)


def simulate_illumina_reads(
    n_reads: int = 1000,
    read_length: int = 75,
    mean_gc: float = 0.5,
    adapter: str = 'AGATCGGAAGAGC',
    adapter_rate: float = 0.05,
    error_rate: float = 0.005,
    seed: int = 42
) -> list[FastqRecord]:
    rng = np.random.default_rng(seed)
    records = []

    # GC-biased base probabilities
    at = (1 - mean_gc) / 2
    gc = mean_gc / 2
    base_probs = [at, gc, gc, at]  # A, C, G, T

    for i in range(n_reads):
        # Insert length: uniform 40–150 bp
        insert_len = int(rng.uniform(40, 150))
        insert = ''.join(rng.choice(['A','C','G','T'], insert_len, p=base_probs))

        # If insert < read_length, add adapter
        if insert_len < read_length:
            seq = insert + adapter[:read_length - insert_len]
        else:
            seq = insert[:read_length]

        # Add sequencing errors
        seq_with_errors = list(seq)
        for pos in range(read_length):
            # Error rate increases toward 3' end
            pos_error_rate = error_rate * (1 + 2 * pos / read_length)
            if rng.random() < pos_error_rate:
                seq_with_errors[pos] = rng.choice([b for b in 'ACGT' if b != seq_with_errors[pos]])
        seq = ''.join(seq_with_errors)

        # Quality scores: high at 5', decreasing at 3'
        qual_scores = []
        for pos in range(read_length):
            base_q = 36 - pos * 0.15  # decreasing mean
            q = int(rng.normal(base_q, 3))
            q = max(2, min(40, q))
            qual_scores.append(q)
        qual_str = ''.join(chr(q + 33) for q in qual_scores)

        records.append(FastqRecord(f'read_{i}', seq[:read_length], qual_str))

    return records


class FastQCReport:
    def __init__(self, records: list[FastqRecord]):
        self.records = records
        self.n = len(records)
        self.read_length = len(records[0].sequence) if records else 0

    def per_base_quality(self) -> np.ndarray:
        all_quals = np.array([[s for s in r.phred_scores] for r in self.records])
        return all_quals.mean(axis=0), np.percentile(all_quals, 25, axis=0), np.percentile(all_quals, 75, axis=0)

    def per_sequence_quality(self) -> tuple[np.ndarray, np.ndarray]:
        means = [r.mean_quality for r in self.records]
        bins = np.arange(0, 42)
        counts, _ = np.histogram(means, bins=bins)
        return bins[:-1], counts

    def gc_distribution(self) -> tuple[np.ndarray, np.ndarray]:
        gcs = [r.gc_content * 100 for r in self.records]
        bins = np.arange(0, 102, 2)
        counts, _ = np.histogram(gcs, bins=bins)
        return bins[:-1], counts

    def adapter_content(self, adapter: str = 'AGATCGGAAGAGC') -> list[float]:
        fracs = []
        for pos in range(self.read_length):
            matches = sum(
                adapter[:min(10, self.read_length-pos)] in r.sequence[pos:pos+10]
                for r in self.records
            )
            fracs.append(matches / self.n)
        return fracs

    def duplication_level(self) -> float:
        seq_counts = Counter(r.sequence for r in self.records)
        duplicated = sum(v for v in seq_counts.values() if v > 1)
        return duplicated / self.n


# Generate and analyse
reads = simulate_illumina_reads(n_reads=500, adapter_rate=0.08)
report = FastQCReport(reads)

mean_q, q25, q75 = report.per_base_quality()
print(f"Per-base mean Q at position 1: {mean_q[0]:.1f}")
print(f"Per-base mean Q at position 75: {mean_q[-1]:.1f}")
print(f"Mean read quality (mean): {np.array([r.mean_quality for r in reads]).mean():.1f}")
print(f"GC content (mean): {np.array([r.gc_content for r in reads]).mean()*100:.1f}%")
print(f"Duplication rate: {report.duplication_level()*100:.1f}%")

In [ ]:
# Step 7 — Visualization: FastQC-style report
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, fig)

positions = np.arange(1, report.read_length + 1)

# Panel A: Per-base quality
ax1 = fig.add_subplot(gs[0, 0])
mean_q, q25, q75 = report.per_base_quality()
ax1.fill_between(positions, q25, q75, alpha=0.3, color='yellow', label='IQR')
ax1.plot(positions, mean_q, 'b-', lw=2, label='Mean Q')
ax1.axhline(30, color='green', ls='--', alpha=0.7, label='Q30')
ax1.axhline(20, color='orange', ls='--', alpha=0.7, label='Q20')
ax1.axhspan(28, 42, alpha=0.05, color='green')
ax1.axhspan(20, 28, alpha=0.05, color='orange')
ax1.axhspan(0, 20, alpha=0.05, color='red')
ax1.set_xlabel('Position in read (bp)')
ax1.set_ylabel('Phred quality score')
ax1.set_title('A. Per-Base Sequence Quality\n(simulated Illumina 75 bp)')
ax1.legend(fontsize=8)
ax1.set_ylim(0, 42)

# Panel B: Per-sequence quality
ax2 = fig.add_subplot(gs[0, 1])
bins, counts = report.per_sequence_quality()
ax2.plot(bins, counts, 'r-', lw=2)
ax2.fill_between(bins, counts, alpha=0.3, color='red')
ax2.set_xlabel('Mean sequence quality')
ax2.set_ylabel('Number of reads')
ax2.set_title('B. Per-Sequence Quality Distribution')
ax2.axvline(30, color='green', ls='--', label='Q30', alpha=0.7)
ax2.legend(fontsize=8)

# Panel C: GC content
ax3 = fig.add_subplot(gs[1, 0])
gc_bins, gc_counts = report.gc_distribution()
ax3.bar(gc_bins, gc_counts, width=2, color='steelblue', alpha=0.7, label='Observed')
# Expected normal distribution centred at mean GC
from scipy.stats import norm
mean_gc_obs = np.array([r.gc_content*100 for r in reads]).mean()
std_gc_obs = np.array([r.gc_content*100 for r in reads]).std()
x_gc = np.linspace(0, 100, 200)
expected = norm.pdf(x_gc, mean_gc_obs, std_gc_obs) * len(reads) * 2
ax3.plot(x_gc, expected, 'r-', lw=2, label='Expected (normal)')
ax3.set_xlabel('% GC content')
ax3.set_ylabel('Number of reads')
ax3.set_title(f'C. GC Content Distribution\n(mean GC={mean_gc_obs:.1f}%)')
ax3.legend(fontsize=8)

# Panel D: Adapter content
ax4 = fig.add_subplot(gs[1, 1])
adapter_fracs = report.adapter_content()
ax4.plot(positions, [f * 100 for f in adapter_fracs], 'r-', lw=2)
ax4.set_xlabel('Position in read (bp)')
ax4.set_ylabel('% reads with adapter')
ax4.set_title('D. Adapter Content\n(should be 0% after trimming)')
ax4.axhline(5, color='orange', ls='--', alpha=0.7, label='5% threshold')
ax4.legend(fontsize=8)

plt.suptitle('FastQC-style Quality Report (Simulated Illumina Data)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fastqc_report.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Quality trimming: sliding window filter
def trim_sliding_window(
    record: FastqRecord,
    window: int = 4,
    min_q: int = 15,
    min_length: int = 36
) -> FastqRecord | None:
    scores = record.phred_scores
    # Find last good position
    end = len(scores)
    for i in range(len(scores) - window, -1, -1):
        window_mean = sum(scores[i:i+window]) / window
        if window_mean >= min_q:
            end = i + window
            break
    # Remove leading low-quality bases
    start = 0
    while start < end and scores[start] < 3:
        start += 1
    if end - start < min_length:
        return None
    return FastqRecord(
        record.header,
        record.sequence[start:end],
        record.quality[start:end]
    )


# Apply trimming
trimmed = [t for r in reads if (t := trim_sliding_window(r)) is not None]
print(f"Before trimming: {len(reads)} reads")
print(f"After trimming: {len(trimmed)} reads ({100*len(trimmed)/len(reads):.1f}% retained)")

lens_before = [len(r.sequence) for r in reads]
lens_after = [len(r.sequence) for r in trimmed]
print(f"Mean read length before: {np.mean(lens_before):.1f} bp")
print(f"Mean read length after:  {np.mean(lens_after):.1f} bp")

---
## Step 8 — Exercises

1. A FastQC report shows per-base Q dropping below 20 at position 50 out of 75.
   What Trimmomatic SLIDINGWINDOW parameter would you use? What are the trade-offs
   of aggressive vs. lenient trimming?
2. Implement `trim_adapter(record, adapter_seq, min_overlap=8)` that removes adapter
   sequence from the 3' end when at least `min_overlap` bp match.
3. A GC content plot shows a bimodal distribution instead of the expected normal.
   What are three possible biological or technical explanations?

---
## Step 10 — Quiz

1. What does a FastQC "FAIL" on adapter content indicate?
2. Why does per-base quality typically drop at the 3' end of Illumina reads?
3. What is the difference between optical duplicates and PCR duplicates?
4. A sequencing run returns reads with 80% GC but the organism has 50% GC. What
   three things would you check?
5. What does MINLEN:36 do in Trimmomatic, and why is this a common minimum?

---
## Step 12 — Reflection

> *[In one paragraph: describe the information in a FastQC per-base quality report
> to someone who has never seen one. What action does a poor report dictate?]*

---
*Next: `03_short_read_alignment_concepts.ipynb`*